# Empathetic Conversational Recommender Systems — Experiment Notebook

Notebook **simple et modulaire** qui re-execute l'experience de l'article :

> Xiaoyu Zhang, Ruobing Xie, Yougang Lyu, Xin Xin, Pengjie Ren, Mingfei Liang, Bo Zhang, Zhanhui Kang, Maarten de Rijke, Zhaochun Ren.  
> **Towards Empathetic Conversational Recommender Systems**. *RecSys '24*, Bari, Italy — [doi:10.1145/3640457.3688133](https://doi.org/10.1145/3640457.3688133).

Le code de reference est celui du depot officiel [zxd-octopus/ECR](https://github.com/zxd-octopus/ECR). Ce notebook se structure comme l'exemple [WebSemantic-Projet-n-1/projet_session/tag_reco_experiment.ipynb](https://github.com/WebSemantic-Projet-n-1/projet_session/blob/main/tag_reco_experiment.ipynb) :

- **Une fonction par cellule** (pas de logique inline).
- **Une seule cellule d'import de donnees**.
- **Tous les graphiques** sont factorises dans `src/viz/plots.py`.
- **Derniere cellule** : compilation des resultats et visualisations comparatives.

### Plan du notebook

1. Parametrages globaux (hyperparametres de l'article, flags d'execution).
2. Preparation de l'environnement (clone du depot ECR + archives `emo_data` / `ckpt`).
3. Pretraitement du dataset **ReDial** (Section 4.1).
4. Sous-tache **Emotional Semantic Fusion** — `train_pre.py` (Section 4.1).
5. Sous-tache **Emotion-aware Item Recommendation** — `train_rec.py` (Section 4.2).
6. Sous-tache **Emotion-aligned Response Generation** — `train_emp.py` + `infer_emp.py` (Section 4.3).
7. Chargement des metriques objectives / subjectives (Tables 1-3).
8. Pipeline principal (import des donnees en **une seule cellule**).
9. Compilation finale + visualisations comparatives.

> **Note pratique.** L'entrainement complet de ECR demande un GPU 24 GB et plusieurs heures. Un drapeau `DRY_RUN` permet de ne lancer que la preparation legere du pipeline tout en reproduisant fidelement les tableaux de l'article (valeurs de repli).

In [1]:
# Cellule 1 - Detection materielle (aligne avec la stack PyTorch de l'article)
import os

FORCE_CPU = False

if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

try:
    import torch
    _torch_ok = True
except ImportError:
    _torch_ok = False

if _torch_ok:
    has_cuda = (not FORCE_CPU) and torch.cuda.is_available()
    device = "cuda" if has_cuda else "cpu"
    print(f"torch {torch.__version__} - device: {device}")
    if has_cuda:
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
        print(f"Memoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    a = torch.randn(1024, 1024, device=device)
    _ = (a @ a).sum().item()
    print(f"Sanity matmul OK sur {device}.")
else:
    print("torch n'est pas installe - installez-le avec `pip install -r requirements.txt` si vous souhaitez re-entrainer les modeles.")

torch 2.11.0a0+a6c236b9fd.nv26.03.46836102 - device: cuda
CUDA device: NVIDIA GeForce RTX 5090
Memoire GPU: 33.7 GB
Sanity matmul OK sur cuda.


In [2]:
# Cellule 2 - Imports (stack notebook + visualisations)
import sys
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.viz.plots import (
    plot_ablation_study,
    plot_emotion_label_distribution,
    plot_feedback_distribution,
    plot_feedback_weights,
    plot_hyperparam_sweep,
    plot_llm_vs_human_correlation,
    plot_model_rankings,
    plot_objective_metrics,
    plot_review_coverage,
    plot_subjective_metrics,
    plot_subjective_radar,
    plot_training_loss,
)

print(f"Project root: {ROOT}")

Project root: /workspace


## 1. Parametrages globaux

Les hyperparametres reprennent la **Section 5.4** de l'article :

- taille d'embedding des labels d'emotion : `48` ;
- seuil de retention `beta = 0.1`, nombre d'emotions partagees `n_f = 3` ;
- triplets / entites injectes dans le prompt : `p_nt = 2`, `p_ne = 4` ;
- **feedback-aware reweighting** (Eq. 7) : `like=2.0`, `dislike=1.0`, `not say=0.5` ;
- batch size `128` pour la recommandation, `16` pour la generation ;
- `lr = 1e-4` (DialoGPT) / `5e-5` (Llama 2-Chat + LoRA).

### Flags d'execution

| Flag | Defaut | Effet |
|---|---|---|
| `dry_run` | `True` | Court-circuite clone / archives / trainings / inference -> juste EDA + tables article. |
| `use_pretrained_ckpt` | `False` | Copie `data/ckpt/{pre,rec,emp}` dans `src_emo/data/saved/` et saute les 3 `train_*.py`. `infer_emp.py` est toujours lance. |
| `batch_scale` | `1.0` | Multiplicateur du `per_device_train_batch_size`. Le `gradient_accumulation_steps` est divise d'autant, donc l'**effective batch reste identique** a l'article. Valeur > 1.0 utile si GPU > 24 GB (ex. RTX 5090 32 GB -> `2.0` a tester). |
| `mixed_precision` | `"no"` | `"bf16"` recommande sur Blackwell / Ampere / Ada (≈ x1.5-2 wall-time, perte negligeable). `"fp16"` sinon. |

Les reglages **par defaut reproduisent fidelement l'article**. Les deux derniers flags sont des leviers hardware sans impact sur le LR ni la semantique du training.

In [3]:
# Cellule 3 - Parametres globaux du pipeline ECR
def get_config():
    return {
        # Chemins principaux
        "root": ROOT,
        "ecr_repo_url": "https://github.com/zxd-octopus/ECR.git",
        "ecr_dir": ROOT / "ECR",
        "src_emo_dir": ROOT / "ECR" / "src_emo",
        "emo_data_archive": ROOT / "data" / "emo_data.zip",
        "emo_data_dir": ROOT / "data" / "emo_data",
        "ckpt_archive": ROOT / "data" / "ckpt.zip",
        "ckpt_dir": ROOT / "data" / "ckpt",
        "results_dir": ROOT / "results",
        # Fichiers de metriques (optionnels : repli sur valeurs de l'article sinon)
        "objective_file": "objective_metrics.csv",
        "subjective_llm_file": "subjective_metrics_llm.csv",
        "subjective_human_file": "subjective_metrics_human.csv",
        "ablation_file": "ablation_metrics.csv",
        # Archives externes (Google Drive - publiees par les auteurs ECR)
        # ~111 MB / ~679 MB : gdown gere automatiquement l'interstitiel > 100 MB.
        "emo_data_gdrive_id": "1fb9kDo8uSRLlwc5c4nUw8DZHR5XOY_l_",
        "ckpt_gdrive_id":     "1uBtcqbQByVrrJ1hEwk2dvsAOxuvEgE19",
        # Hyperparametres article (Section 5.4)
        "emotion_embed_dim": 48,
        "beta": 0.1,
        "n_f": 3,
        "p_nt": 2,
        "p_ne": 4,
        "feedback_weights": {"like": 2.0, "dislike": 1.0, "not say": 0.5},
        "lr_dialogpt": 1e-4,
        "lr_llama": 5e-5,
        "batch_rec": 128,
        "batch_gen": 16,
        "seed": 42,
        # Flags d'execution
        "dry_run": False,              # True  = aucun training/inference, juste EDA + tables article
        "use_pretrained_ckpt": False,  # True  = court-circuite les 3 trainings avec les poids data/ckpt/*
                                       #         (utile pour ne relancer que infer_emp.py et gagner >20h GPU)
        "skip_clone": False,
        "skip_download": False,
        # Optimisations hardware (profite d'un GPU > 24 GB comme la RTX 5090 32 GB).
        # L'article a ete entraine sur 24 GB : valeurs par defaut => fidelite article.
        # batch_scale > 1.0 : augmente `per_device_train_batch_size` et divise
        # `gradient_accumulation_steps` par le meme facteur. L'effective batch
        # reste identique a celui de l'article -> pas besoin de retuner le LR.
        # mixed_precision: "no" (article), "bf16" (recommande Blackwell), "fp16".
        "batch_scale": 2.0,
        "mixed_precision": "bf16",
        "num_workers": 16,
        "enable_torch_compile": True,
        # Modeles de base requis par les scripts ECR (Section 4) :
        # `AutoTokenizer.from_pretrained("save/dialogpt/")` etc. -> repertoires locaux.
        "base_models": {
            "microsoft/DialoGPT-small": "save/dialogpt",
            "roberta-base":             "save/roberta",
        },
    }


def accelerate_launch_cmd(cfg, script):
    """Construit `accelerate launch [--mixed_precision X] script` selon la config."""
    cmd = ["accelerate", "launch"]
    mp = cfg.get("mixed_precision", "no")
    if mp and mp != "no":
        cmd += ["--mixed_precision", mp]
    cmd.append(script)
    return cmd


def scaled_batch(cfg, per_device_batch, grad_accum):
    """Scale per_device_batch par cfg['batch_scale'] en reduisant grad_accum d'autant.

    Preserve l'effective batch = per_device_batch * grad_accum, donc le LR
    reste valide sans retuning.
    """
    scale = max(1.0, float(cfg.get("batch_scale", 1.0)))
    if scale == 1.0:
        return per_device_batch, grad_accum
    effective = per_device_batch * grad_accum
    new_batch = max(1, int(round(per_device_batch * scale)))
    new_grad_accum = max(1, effective // new_batch)
    return new_batch, new_grad_accum


cfg = get_config()
cfg

{'root': PosixPath('/workspace'),
 'ecr_repo_url': 'https://github.com/zxd-octopus/ECR.git',
 'ecr_dir': PosixPath('/workspace/ECR'),
 'src_emo_dir': PosixPath('/workspace/ECR/src_emo'),
 'emo_data_archive': PosixPath('/workspace/data/emo_data.zip'),
 'emo_data_dir': PosixPath('/workspace/data/emo_data'),
 'ckpt_archive': PosixPath('/workspace/data/ckpt.zip'),
 'ckpt_dir': PosixPath('/workspace/data/ckpt'),
 'results_dir': PosixPath('/workspace/results'),
 'objective_file': 'objective_metrics.csv',
 'subjective_llm_file': 'subjective_metrics_llm.csv',
 'subjective_human_file': 'subjective_metrics_human.csv',
 'ablation_file': 'ablation_metrics.csv',
 'emo_data_gdrive_id': '1fb9kDo8uSRLlwc5c4nUw8DZHR5XOY_l_',
 'ckpt_gdrive_id': '1uBtcqbQByVrrJ1hEwk2dvsAOxuvEgE19',
 'emotion_embed_dim': 48,
 'beta': 0.1,
 'n_f': 3,
 'p_nt': 2,
 'p_ne': 4,
 'feedback_weights': {'like': 2.0, 'dislike': 1.0, 'not say': 0.5},
 'lr_dialogpt': 0.0001,
 'lr_llama': 5e-05,
 'batch_rec': 128,
 'batch_gen': 16,
 '

## 2. Preparation de l'environnement

Nous clonons le depot [zxd-octopus/ECR](https://github.com/zxd-octopus/ECR) puis telechargeons les deux archives publiees par les auteurs sur Google Drive :

- [`emo_data.zip`](https://drive.google.com/file/d/1fb9kDo8uSRLlwc5c4nUw8DZHR5XOY_l_/view) (~111 MB) — donnees emotionnelles pretraitees (annotations GPT-3.5 + reviews IMDb filtrees — Section 4.1). Contient `llama_train.json`, `llama_test.json`, les entites DBpedia et les reviews retenues.
- [`ckpt.zip`](https://drive.google.com/file/d/1uBtcqbQByVrrJ1hEwk2dvsAOxuvEgE19/view) (~679 MB) — poids entraines (GPT-2 classifieur d'emotions + ECR[DialoGPT] + ECR[Llama 2-Chat]).

Le telechargement utilise [`gdown`](https://github.com/wkentaro/gdown) qui gere automatiquement l'interstitiel Google Drive "virus scan warning" servi pour tout fichier > 100 MB. En cas d'echec reseau, deposer manuellement les deux `.zip` dans `data/` puis relancer la cellule.

In [4]:
# Cellule 4 - Clone du depot officiel ECR
def clone_ecr_repo(cfg):
    """Clone le depot zxd-octopus/ECR a la racine du projet (idempotent)."""
    if cfg["skip_clone"] or cfg["ecr_dir"].exists():
        print(f"ECR repo deja present: {cfg['ecr_dir']}")
        return cfg["ecr_dir"]
    print(f"Clonage de {cfg['ecr_repo_url']} -> {cfg['ecr_dir']}")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", cfg["ecr_repo_url"], str(cfg["ecr_dir"])],
        capture_output=True,
        text=True,
        check=False,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("[STDERR]", result.stderr)
    return cfg["ecr_dir"]

### 2ter. Patches de compatibilite stack moderne

Le code ECR a ete ecrit pour `transformers 4.15` / `pyg 2.0.1` (fin 2023). Plusieurs APIs ont bouge depuis :

| Symbole | Statut | Correctif |
|---|---|---|
| `transformers.AdamW` | Retire en 4.30+ | -> `torch.optim.AdamW` (impl fusee CUDA plus rapide) |
| `transformers.modeling_utils.find_pruneable_heads_and_indices` | Deplace 4.17 | -> `transformers.pytorch_utils` |
| `transformers.utils.model_parallel_utils` | Retire 4.40+ | Stubs no-op (GPT-2 `parallelize()` non utilise par ECR) |
| `pyg.typing.SparseTensor` | Requiert `torch-sparse` | Format standard Tensor[2,E] + edge_type separe (compat RGCNConv) |

`patch_ecr_compat(cfg)` applique 5 patches idempotents sur les fichiers clones dans `ECR/src_emo/`. Aucun PR upstream ni download externe requis.

In [5]:
# Cellule 4b - Compatibilite stack moderne (transformers >= 4.40 / pyg >= 2.5 / accelerate >= 0.20 / torch >= 2)
# ECR a ete ecrit pour transformers 4.15 / pyg 2.0.1 / accelerate 0.8 / torch 1.8.
# On applique des patches idempotents aux fichiers clones. Aucune PR upstream requise ;
# un `cd ECR && git checkout -- src_emo/` restaure l'original et patch_ecr_compat
# se re-applique au run suivant.
import re as _re


def _patch_file(path, replacements, description):
    if not path.exists():
        print(f"[patch] SKIP {description}: {path} absent")
        return
    text = path.read_text()
    original = text
    for pat, repl in replacements:
        text = _re.sub(pat, repl, text, flags=_re.MULTILINE)
    if text == original:
        print(f"[patch] {description}: deja applique (idempotent)")
        return
    path.write_text(text)
    print(f"[patch] {description}: OK")


def patch_ecr_compat(cfg):
    """Rend le code ECR compatible avec transformers >= 4.40 et pyg >= 2.5.

    Patches appliques (tous idempotents) :
      1. `transformers.AdamW` retire -> `torch.optim.AdamW` (train_{pre,rec,emp}.py)
      2. `modeling_utils.find_pruneable_heads_and_indices` -> `pytorch_utils`
      3. `model_parallel_utils` retire en 4.40 -> stubs no-op (jamais appele)
      4. `SparseTensor` (pyg >= 2.5 exige torch-sparse) -> Tensor [2,E] + edge_type
      5. RGCNConv call dans model_prompt.py -> forme standard (x, edge_index, edge_type)
      6. `Accelerator(..., fp16=...)` retire en accelerate 0.12 -> drop kwarg (mixed_precision via `accelerate launch`)
      7. `torch.set_deterministic` retire en torch 2.0 -> `torch.use_deterministic_algorithms(True, warn_only=True)`
         + CUDA_LAUNCH_BLOCKING / CUBLAS_WORKSPACE_CONFIG commentes (serialisation kernels -> 5-10x plus lent)
      8. `accelerator.use_fp16` retire en accelerate 0.20 -> getattr fallback (bf16 via launch)
      9. `.cuda()` hardcode dans dataset_dbpedia.py -> `.to(device)` pour tolerer CPU transitoire
     10. `print("here")` debug de l'auteur supprime (5 locations : train_pre/dataset_emp/imdb_filter/co_appear)
     11. `evaluate_rec.py` cree `save/redial_rec/` avant ecriture de `rec.json`
     12. `transformers.file_utils.ModelOutput` deprecated 4.25 (retire 5.0) -> `transformers.utils.ModelOutput`
     13. `PromptGPT2forCRS` herite aussi de `GenerationMixin` (transformers >= 4.50)
     14. `train_rec.py` ne force plus `CUDA_VISIBLE_DEVICES=3`
     15. `log/` est cree avant `logger.add(...)` dans train_{pre,rec,emp}.py + infer_emp.py
     16. chemins de sortie stabilises (`pre`, `rec`, `emp`) sans suffixe timestamp
     17. `train_pre.py` sauvegarde par defaut dans `data/saved/pre`
     18. `train_emp.py` sauvegarde par defaut dans `data/saved/emp`
     19. `infer_emp.py` charge localement (`local_files_only=True`) si le path modele existe
     20. suppression de `retain_graph=True` dans train_pre/train_rec
    """
    src_emo = cfg["src_emo_dir"]
    if not src_emo.exists():
        print(f"src_emo absent ({src_emo}) -> patches ignores.")
        return

    # Patch 1 : AdamW dans les 3 scripts de training
    adamw_patch = [(
        r"from transformers import AdamW, get_linear_schedule_with_warmup, AutoTokenizer, AutoModel",
        "from transformers import get_linear_schedule_with_warmup, AutoTokenizer, AutoModel\nfrom torch.optim import AdamW",
    )]
    for name in ("train_pre.py", "train_rec.py", "train_emp.py"):
        _patch_file(src_emo / name, adamw_patch, description=f"AdamW -> torch.optim ({name})")

    # Patches 2 + 3 : model_gpt2.py
    _patch_file(src_emo / "model_gpt2.py", [
        (
            r"from transformers\.modeling_utils import find_pruneable_heads_and_indices, prune_conv1d_layer",
            "from transformers.pytorch_utils import find_pruneable_heads_and_indices, prune_conv1d_layer",
        ),
        (
            r"^from transformers\.utils\.model_parallel_utils import assert_device_map, get_device_map\s*$",
            (
                "try:\n"
                "    from transformers.utils.model_parallel_utils import assert_device_map, get_device_map\n"
                "except ImportError:  # transformers >= 4.40 a retire le parallelisme custom GPT-2\n"
                "    def assert_device_map(*a, **kw): return None\n"
                "    def get_device_map(*a, **kw): return {}\n"
            ),
        ),
    ], description="model_gpt2.py imports")

    # Patch 4 : SparseTensor dans dataset_dbpedia.py
    _patch_file(src_emo / "dataset_dbpedia.py", [(
        r"^(\s*)self\.edge_index = SparseTensor\(row=self\.edge_index\[0\], col=self\.edge_index\[1\], value=self\.edge_type\)\s*$",
        r"\1# PATCH: SparseTensor supprime (PyG >= 2.5 requiert torch-sparse).\n"
        r"\1# edge_index reste Tensor [2,E], edge_type reste Tensor [E] -> format PyG standard.\n"
        r"\1# (voir model_prompt.py pour l'appel RGCNConv correspondant)",
    )], description="dataset_dbpedia.py SparseTensor")

    # Patch 5 : RGCNConv call dans model_prompt.py - on passe a la forme standard (3 args)
    _patch_file(src_emo / "model_prompt.py", [(
        r"entity_embeds = self\.kg_encoder\(node_embeds, self\.edge_index\) \+ node_embeds.*?\n\s*#entity_embeds = self\.kg_encoder\(node_embeds, self\.edge_index, self\.edge_type\) \+ node_embeds",
        "entity_embeds = self.kg_encoder(node_embeds, self.edge_index, self.edge_type) + node_embeds",
    )], description="model_prompt.py RGCNConv call")

    # Patch 6 : Accelerator(fp16=...) retire en accelerate 0.12+
    #           On s'appuie sur `accelerate launch --mixed_precision <mp>` pour la config.
    accel_patch = [(
        r"accelerator = Accelerator\(device_placement=False, fp16=args\.fp16\)",
        "accelerator = Accelerator(device_placement=False)  # PATCH: mixed_precision via `accelerate launch`",
    )]
    for name in ("train_pre.py", "train_rec.py", "train_emp.py", "infer_emp.py"):
        _patch_file(src_emo / name, accel_patch, description=f"Accelerator fp16 kwarg ({name})")

    # Patch 7 : seed_torch dans config.py
    #   - torch.set_deterministic -> torch.use_deterministic_algorithms (retire en torch 2.0)
    #   - CUDA_LAUNCH_BLOCKING=1 et CUBLAS_WORKSPACE_CONFIG:4096:8 : residus debug qui
    #     serialisent les kernels CUDA et coutent 5-10x en wall-time sur GPU moderne.
    _patch_file(src_emo / "config.py", [
        (
            r"torch\.set_deterministic\(True\)",
            "torch.use_deterministic_algorithms(True, warn_only=True)  # PATCH: set_deterministic retire en torch 2.0",
        ),
        (
            r'^(\s*)os\.environ\["CUDA_LAUNCH_BLOCKING"\] = "1"\s*$',
            r'\1# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # PATCH: desactive (5-10x plus lent sur GPU moderne)',
        ),
        (
            r'^(\s*)os\.environ\["CUBLAS_WORKSPACE_CONFIG"\] = ":4096:8"\s*$',
            r'\1# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # PATCH: desactive (lie a determinisme strict)',
        ),
    ], description="config.py seed_torch")

    # Patch 8 : accelerator.use_fp16 retire en accelerate 0.20
    #   Les collators lisent ce flag pour activer `torch.cuda.amp.autocast` manuellement ;
    #   comme on passe --mixed_precision bf16 via `accelerate launch`, l'autocast est deja
    #   gere et on peut renvoyer False (fallback sur bf16 automatique).
    use_fp16_patch = [(
        r"accelerator\.use_fp16",
        "getattr(accelerator, 'use_fp16', False)",
    )]
    for name in ("train_pre.py", "train_emp.py", "infer_emp.py"):
        _patch_file(src_emo / name, use_fp16_patch, description=f"accelerator.use_fp16 ({name})")

    # Patch 9 : .cuda() hardcode dans dataset_dbpedia.py
    #   Tolere un fallback CPU si aucun GPU n'est detecte au moment de construire DBpedia
    #   (cas transitoire : process zombie tenant de la VRAM, driver en power-save, ...).
    #   RGCNConv gere ensuite le deplacement vers le device du modele lors du forward.
    _patch_file(src_emo / "dataset_dbpedia.py", [
        (
            r"^([ \t]+)self\.edge_index = edge\[:, :2\]\.t\(\)\.cuda\(\)$",
            r"\1_dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n\1self.edge_index = edge[:, :2].t().to(_dev)  # PATCH: tolere CPU",
        ),
        (
            r"^([ \t]+)self\.edge_type = edge\[:, 2\]\.cuda\(\)$",
            r"\1self.edge_type = edge[:, 2].to(_dev)  # PATCH: tolere CPU",
        ),
    ], description="dataset_dbpedia.py .cuda() hardcode")

    # Patch 10 : prints de debug `print("here")` laisses par les auteurs
    #   dataset_emp.py (2 occurrences), imdb_review_entity_filter.py, train_pre.py, co_appear.py
    debug_patch = [(
        r"^(\s*)print\(([\"'])here\2\)\s*$",
        r'\1pass  # PATCH: debug print("here") supprime',
    )]
    for name in ("dataset_emp.py", "imdb_review_entity_filter.py", "train_pre.py", "co_appear.py"):
        _patch_file(src_emo / name, debug_patch, description=f'debug print("here") dans {name}')

    # Patch 11 : ensure save/redial_rec existe avant ecriture rec.json
    _patch_file(src_emo / "evaluate_rec.py", [(
        r'^(\s*)self\.log_file = open\("save/redial_rec/rec\.json", ["\']w["\'], buffering=1\)\s*$',
        r'\1import os\n\1os.makedirs("save/redial_rec", exist_ok=True)  # PATCH: cree le dossier de logs\n\1self.log_file = open("save/redial_rec/rec.json", "w", buffering=1)',
    )], description="evaluate_rec.py rec.json directory")

    # Patch 12 : transformers.file_utils -> transformers.utils (preemptif)
    #   file_utils est un alias deprecie depuis 4.25 et retire en 5.0.
    #   Ca marche encore en 4.57 mais emet un FutureWarning.
    _patch_file(src_emo / "model_gpt2.py", [(
        r"from transformers\.file_utils import ModelOutput",
        "from transformers.utils import ModelOutput  # PATCH: file_utils deprecated -> utils",
    )], description="file_utils -> utils (model_gpt2.py)")
    # Patch 21 : train_rec.py charge par defaut le checkpoint pre/final
    _patch_file(src_emo / "train_rec.py", [(
        r'parser\.add_argument\("--prompt_encoder", type=str,default="data/saved/pre/"\)',
        'parser.add_argument("--prompt_encoder", type=str,default="data/saved/pre/final/")',
    )], description="train_rec.py prompt_encoder default")

    # Patch 22 : merge_rec.py explicite les cas rec.json vide/incomplet
    _patch_file(src_emo / "data" / "redial_gen" / "merge_rec.py", [
        (
            r'gen_data = gen_file\.readlines\(\)',
            (
                "gen_data = gen_file.readlines()\n"
                "if len(gen_data) == 0:\n"
                "    raise RuntimeError(\n"
                "        f\"{gen_file_path} est vide: train_rec n'a pas produit de predictions.\"\n"
                "    )"
            ),
        ),
        (
            r"for i in range\(len\(raw\['rec'\]\)\):\n\s*gen = json\.loads\(gen_data\[cnt\]\)",
            (
                "for i in range(len(raw['rec'])):\n"
                "                if cnt >= len(gen_data):\n"
                "                    raise RuntimeError(\n"
                "                        f\"Nombre de predictions insuffisant dans {gen_file_path}: \"\n"
                "                        f\"{len(gen_data)} lignes, au moins {cnt + 1} requises.\"\n"
                "                    )\n"
                "                gen = json.loads(gen_data[cnt])"
            ),
        ),
    ], description="merge_rec.py guardrails")


    # Patch 23 : PromptGPT2forCRS herite de GenerationMixin (transformers >= 4.50)
    _patch_file(src_emo / "model_gpt2.py", [
        (
            r'^from transformers import Conv1D\s*$',
            "from transformers import Conv1D\nfrom transformers.generation import GenerationMixin",
        ),
        (
            r'^class PromptGPT2forCRS\(GPT2PreTrainedModel\):\s*$',
            "class PromptGPT2forCRS(GPT2PreTrainedModel, GenerationMixin):",
        ),
    ], description="model_gpt2.py GenerationMixin")

    # Patch 24 : train_rec.py n'impose pas CUDA_VISIBLE_DEVICES=3
    _patch_file(src_emo / "train_rec.py", [(
        r'^os\.environ\["CUDA_VISIBLE_DEVICES"\] = "3"\s*$',
        "# PATCH: suppression CUDA_VISIBLE_DEVICES hardcode",
    )], description="train_rec.py CUDA_VISIBLE_DEVICES")

    # Patch 25 : dossiers log/ assures avant logger.add
    for name in ("train_pre.py", "train_rec.py", "train_emp.py", "infer_emp.py"):
        _patch_file(src_emo / name, [(
            r'^(\s*)local_time = time\.strftime\("%Y-%m-%d-%H-%M-%S", time\.localtime\(\)\)\s*$',
            r'\1local_time = time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())\n\1os.makedirs("log", exist_ok=True)',
        )], description=f"{name} log directory")

    # Patch 26 : sorties stables (sans suffixe timestamp) pour chainage pre->rec et emp->infer
    _patch_file(src_emo / "train_pre.py", [
        (
            r'parser\.add_argument\("--output_dir", type=str, default=\'data/saved/pre-trained\'',
            "parser.add_argument(\"--output_dir\", type=str, default=\'data/saved/pre\'",
        ),
        (
            r'^(\s*)args\.output_dir = args\.output_dir \+ local_time\s*$',
            r'\1# PATCH: keep stable output dir (no timestamp)',
        ),
    ], description="train_pre.py stable output_dir")

    _patch_file(src_emo / "train_emp.py", [
        (
            r'parser\.add_argument\("--output_dir", type=str, help="Where to store the final model\.",default="data/saved/emp_conv"\)',
            'parser.add_argument("--output_dir", type=str, help="Where to store the final model.", default="data/saved/emp")',
        ),
        (
            r'^(\s*)args\.output_dir = args\.output_dir\+local_time\s*$',
            r'\1# PATCH: keep stable output dir (no timestamp)',
        ),
    ], description="train_emp.py stable output_dir")

    _patch_file(src_emo / "train_rec.py", [(
        r'^(\s*)args\.output_dir = args\.output_dir \+ local_time\s*$',
        r'\1# PATCH: keep stable output dir (no timestamp)',
    )], description="train_rec.py stable output_dir")

    # Patch 27 : infer_emp.py charge local_files_only si le modele est un dossier local
    _patch_file(src_emo / "infer_emp.py", [(
        r'^(\s*)model = PromptGPT2forCRS\.from_pretrained\(args\.model\)\s*$',
        r'\1if os.path.isdir(args.model):\n\1    model = PromptGPT2forCRS.from_pretrained(args.model, local_files_only=True)\n\1else:\n\1    model = PromptGPT2forCRS.from_pretrained(args.model)',
    )], description="infer_emp.py local model loading")

    # Patch 28 : suppress retain_graph=True (memoire inutile)
    for name in ("train_pre.py", "train_rec.py"):
        _patch_file(src_emo / name, [(
            r'accelerator\.backward\(loss, retain_graph = True\)',
            'accelerator.backward(loss)',
        )], description=f"{name} retain_graph")


In [6]:
# Cellule 5 - Telechargement des archives emo_data / ckpt depuis Google Drive
# gdown est necessaire : Google Drive sert une page HTML d'interstitiel pour
# tout fichier > 100 MB (ckpt.zip pese 679 MB). curl recupererait alors le
# HTML au lieu du .zip. gdown detecte cette page et suit le token `confirm`.
import gdown


def download_gdrive(file_id, dst, description):
    """Telecharge une archive depuis Google Drive (gere l'interstitiel > 100 MB)."""
    dst = Path(dst)
    if dst.exists() and dst.stat().st_size > 1024:
        print(f"[{description}] deja present: {dst} ({dst.stat().st_size / 1e6:.1f} MB)")
        return dst
    dst.parent.mkdir(parents=True, exist_ok=True)
    print(f"[{description}] gdown download (id={file_id}) -> {dst}")
    try:
        out = gdown.download(id=file_id, output=str(dst), quiet=False)
    except Exception as exc:
        print(f"[{description}] ECHEC gdown: {exc}")
        return None
    if out is None or not dst.exists() or dst.stat().st_size < 1024:
        print(f"[{description}] telechargement vide, deposer manuellement dans: {dst}")
        if dst.exists():
            dst.unlink(missing_ok=True)
        return None
    print(f"[{description}] OK ({dst.stat().st_size / 1e6:.1f} MB)")
    return dst


def unzip_if_needed(archive, dst_dir):
    """Decompresse l'archive si necessaire."""
    dst_dir = Path(dst_dir)
    if dst_dir.exists() and any(dst_dir.iterdir()):
        print(f"  {dst_dir} deja extrait.")
        return dst_dir
    if archive is None or not Path(archive).exists():
        print(f"  archive manquante, extraction ignoree.")
        return None
    dst_dir.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["unzip", "-oq", str(archive), "-d", str(dst_dir)],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        print("[STDERR]", result.stderr)
    print(f"  extrait -> {dst_dir}")
    return dst_dir


def download_external_assets(cfg):
    """Telecharge + extrait emo_data.zip et ckpt.zip depuis Google Drive."""
    if cfg["skip_download"]:
        print("skip_download=True -> archives non recuperees.")
        return
    download_gdrive(cfg["emo_data_gdrive_id"], cfg["emo_data_archive"], "emo_data")
    unzip_if_needed(cfg["emo_data_archive"], cfg["emo_data_dir"])
    download_gdrive(cfg["ckpt_gdrive_id"], cfg["ckpt_archive"], "ckpt")
    unzip_if_needed(cfg["ckpt_archive"], cfg["ckpt_dir"])

### 2bis. Modeles de base HF + poids pre-entraines (optionnel)

Les scripts ECR chargent DialoGPT-small et RoBERTa-base via des chemins **locaux** (`save/dialogpt/`, `save/roberta/`) au sein de `src_emo/`. On les telecharge une fois via `huggingface_hub.snapshot_download` — les caches Docker Compose (`hf_cache`) evitent tout re-download entre runs.

`install_pretrained_ckpts` est active **uniquement** si le flag `use_pretrained_ckpt=True`. Il copie :

- `data/ckpt/pre/`  -> `src_emo/data/saved/pre/`   (Emotional Semantic Fusion)
- `data/ckpt/rec/`  -> `src_emo/data/saved/rec/`   (Emotion-aware Recommendation)
- `data/ckpt/emp/`  -> `src_emo/data/saved/emp/`   (Emotion-aligned DialoGPT)

Ainsi `train_rec.py` (qui lit `--prompt_encoder data/saved/pre/`) et `infer_emp.py` (qui lit `--model data/saved/emp/`) trouvent les poids directement. Les trois cellules `run_*` respectent ce flag : elles sautent proprement le training correspondant. Passer `use_pretrained_ckpt=False` relance le pipeline complet sans code a modifier.

In [7]:
# Cellule 5b - Modeles de base HF + poids pre-entraines (optionnel)
# Les scripts ECR appellent `AutoTokenizer.from_pretrained("save/dialogpt/")` et
# `AutoModel.from_pretrained("save/roberta/")` : ces repertoires doivent exister
# dans src_emo/. On telecharge depuis HF Hub une fois (cache volume HF partage).
from huggingface_hub import snapshot_download


def install_base_models(cfg):
    """Telecharge DialoGPT-small + RoBERTa-base dans src_emo/save/ (HF snapshots)."""
    src_emo = cfg["src_emo_dir"]
    for hf_id, rel_dir in cfg["base_models"].items():
        local = src_emo / rel_dir
        if (local / "config.json").exists():
            print(f"[base] {hf_id} deja present: {local}")
            continue
        local.mkdir(parents=True, exist_ok=True)
        print(f"[base] snapshot_download {hf_id} -> {local}")
        try:
            snapshot_download(
                repo_id=hf_id,
                local_dir=str(local),
                local_dir_use_symlinks=False,
            )
        except TypeError:
            # API huggingface_hub >= 0.23 a retire local_dir_use_symlinks
            snapshot_download(repo_id=hf_id, local_dir=str(local))
        print(f"[base] {hf_id} OK ({sum(1 for _ in local.iterdir())} fichiers)")


def install_pretrained_ckpts(cfg):
    """Copie data/ckpt/{pre,rec,emp}/* dans src_emo/data/saved/{pre,rec,emp}/.

    Active seulement si cfg["use_pretrained_ckpt"] est True.
    Permet aux scripts train_rec.py (qui lit --prompt_encoder data/saved/pre/)
    et infer_emp.py (qui lit --model data/saved/emp/) de trouver les poids sans
    re-entrainement. Aucun fichier n'est supprime : on peut toujours repartir
    sur un training from-scratch en desactivant le flag.
    """
    if not cfg.get("use_pretrained_ckpt"):
        print("use_pretrained_ckpt=False -> poids fournis non installes (training complet).")
        return
    src_emo = cfg["src_emo_dir"]
    ckpt = cfg["ckpt_dir"]
    saved = src_emo / "data" / "saved"
    saved.mkdir(parents=True, exist_ok=True)
    for sub in ("pre", "rec", "emp"):
        src = ckpt / sub
        dst = saved / sub
        if not src.exists():
            print(f"[ckpt] source absente: {src}")
            continue
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.iterdir():
            if f.name.startswith(".") or f.name == "__MACOSX":
                continue
            target = dst / f.name
            if target.exists():
                continue
            if f.is_file():
                shutil.copy2(f, target)
            else:
                shutil.copytree(f, target)
        print(f"[ckpt] {src} -> {dst}")

## 3. Pretraitement du dataset ReDial (Section 4.1)

Le README du depot ECR enchaine les etapes suivantes :

```bash
cd src_emo
cp -r data/emo_data/* data/redial/
python data/redial/process.py
```

`process.py` integre les annotations GPT-3.5 (9 labels d'emotion : *like, curious, happy, grateful, negative, neutral, nostalgia, agreement, surprise* — cf. Section 4.1.1) au corpus ReDial brut et fournit les entites DBpedia alignees. Pour la recommandation, `process_mask.py` masque l'item cible ; `merge.py` recolte les predictions du modele conversationnel UniCRS. La fonction ci-dessous orchestre ces commandes.

In [8]:
# Cellule 6 - Pretraitement du dataset ReDial (Section 4.1)
def _run(cmd, cwd=None, description=""):
    """Wrapper subprocess avec affichage compact."""
    print(f"[run] {description} :: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, check=False)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        print("[STDERR]", result.stderr.strip())
    return result.returncode == 0


def prepare_redial_data(cfg):
    """Reproduit la preparation `data/redial/` et `data/redial_gen/` du README ECR."""
    src_emo = cfg["src_emo_dir"]
    if not src_emo.exists():
        print(f"src_emo indisponible: {src_emo} -> clone le depot d'abord.")
        return False
    redial = src_emo / "data" / "redial"
    redial_gen = src_emo / "data" / "redial_gen"
    emo = cfg["emo_data_dir"]

    redial.mkdir(parents=True, exist_ok=True)
    redial_gen.mkdir(parents=True, exist_ok=True)

    if emo.exists():
        for p in emo.iterdir():
            tgt = redial / p.name
            if not tgt.exists():
                (shutil.copy2 if p.is_file() else shutil.copytree)(p, tgt)
    else:
        print(f"emo_data non trouve ({emo}) - les scripts supposent qu'il a ete decompresse.")

    ok = True
    if cfg["dry_run"]:
        print("DRY_RUN=True -> scripts process.py / process_mask.py / merge.py non executes.")
        return True

    # Les scripts ECR utilisent des chemins relatifs (ex: open('entity2id.json')) :
    # ils doivent etre lances depuis leur propre repertoire, pas depuis src_emo.
    ok &= _run(["python", "process.py"],      cwd=redial,     description="process.py")
    ok &= _run(["python", "process_mask.py"], cwd=redial,     description="process_mask.py")
    if redial.exists() and redial_gen.exists():
        for p in redial.iterdir():
            tgt = redial_gen / p.name
            if not tgt.exists():
                (shutil.copy2 if p.is_file() else shutil.copytree)(p, tgt)
    ok &= _run(["python", "merge.py"],        cwd=redial_gen, description="merge.py")
    return ok

## 4. Sous-tache Emotional Semantic Fusion (Section 4.1)

La fusion emotionnelle est entrainee avec `train_pre.py`. L'objectif est de fusionner la semantique des mots (token-level) et des entites emotionnelles (utterance-level) avant la phase de recommandation. Les hyperparametres du depot sont :

- `num_train_epochs=10`, `gradient_accumulation_steps=4`, `per_device_train_batch_size=16` ;
- `max_length=200`, `prompt_max_length=200`, `entity_max_length=32` ;
- `learning_rate=5e-4`, `num_warmup_steps=1389`.

L'option `--nei_mer` active la fusion emotion/neighbour decrite Section 4.1.

In [9]:
# Cellule 7 - Emotional Semantic Fusion (train_pre.py)
def run_emotional_semantic_fusion(cfg):
    """Execute `accelerate launch train_pre.py ...` (Section 4.1)."""
    if cfg["dry_run"]:
        print("DRY_RUN=True -> train_pre.py non lance (>6h sur GPU 24GB).")
        return None
    if cfg.get("use_pretrained_ckpt"):
        print("use_pretrained_ckpt=True -> train_pre.py ignore (poids charges depuis data/ckpt/pre/).")
        return True
    # Article: per_device_batch=16, grad_accum=4 -> effective batch 64.
    batch, grad_accum = scaled_batch(cfg, per_device_batch=16, grad_accum=4)
    cmd = accelerate_launch_cmd(cfg, "train_pre.py") + [
        "--dataset", "redial",
        "--num_train_epochs", "10",
        "--gradient_accumulation_steps", str(grad_accum),
        "--per_device_train_batch_size", str(batch),
        "--per_device_eval_batch_size", "64",
        "--num_warmup_steps", "1389",
        "--max_length", "200",
        "--prompt_max_length", "200",
        "--entity_max_length", "32",
        "--learning_rate", "5e-4",
        "--seed", str(cfg["seed"]),
        "--num_workers", str(cfg.get("num_workers", 16)),
        "--nei_mer",
    ]
    if cfg.get("enable_torch_compile", False):
        cmd.append("--enable_torch_compile")
    return _run(cmd, cwd=cfg["src_emo_dir"], description="train_pre")


## 5. Sous-tache Emotion-aware Item Recommendation (Section 4.2)

Cette etape entraine le module de recommandation avec :

- representations locales / globales emotion-aware (Eq. 3-6) ;
- **feedback-aware item reweighting** (Eq. 7) avec les poids `like=2.0`, `dislike=1.0`, `not say=0.5` ;
- option `--use_sentiment` pour utiliser la classification fine-tunee GPT-2 du README.

In [10]:
# Cellule 8 - Emotion-aware Item Recommendation (train_rec.py)
def run_recommendation_training(cfg):
    if cfg["dry_run"]:
        print("DRY_RUN=True -> train_rec.py non lance (entrainement GPU necessaire).")
        return None
    if cfg.get("use_pretrained_ckpt"):
        print("use_pretrained_ckpt=True -> train_rec.py ignore (poids charges depuis data/ckpt/rec/).")
        return True
    pre_ckpt_candidates = [
        cfg["src_emo_dir"] / "data" / "saved" / "pre" / "final" / "model.pt",
        cfg["src_emo_dir"] / "data" / "saved" / "pre" / "model.pt",
    ]
    pre_ckpt = next((p for p in pre_ckpt_candidates if p.exists()), None)
    if pre_ckpt is None:
        print("[skip] train_rec: checkpoint pre absent (attendus: data/saved/pre/final/model.pt ou data/saved/pre/model.pt).")
        return False
    prompt_encoder_dir = str(pre_ckpt.parent.relative_to(cfg["src_emo_dir"]))
    # Article: per_device_batch=16 (=batch_rec//8), grad_accum=8 -> effective batch 128.
    batch, grad_accum = scaled_batch(cfg, per_device_batch=cfg["batch_rec"] // 8, grad_accum=8)
    cmd = accelerate_launch_cmd(cfg, "train_rec.py") + [
        "--dataset", "redial_gen",
        "--n_prefix_rec", "10",
        "--num_train_epochs", "5",
        "--per_device_train_batch_size", str(batch),
        "--per_device_eval_batch_size", "32",
        "--gradient_accumulation_steps", str(grad_accum),
        "--num_warmup_steps", "530",
        "--context_max_length", "200",
        "--prompt_max_length", "200",
        "--entity_max_length", "32",
        "--prompt_encoder", prompt_encoder_dir,
        "--learning_rate", str(cfg["lr_dialogpt"]),
        "--seed", "8",
        "--num_workers", str(cfg.get("num_workers", 16)),
        "--like_score", str(cfg["feedback_weights"]["like"]),
        "--dislike_score", str(cfg["feedback_weights"]["dislike"]),
        "--notsay_score", str(cfg["feedback_weights"]["not say"]),
        "--weighted_loss",
        "--nei_mer",
        "--use_sentiment",
    ]
    if cfg.get("enable_torch_compile", False):
        cmd.append("--enable_torch_compile")
    return _run(cmd, cwd=cfg["src_emo_dir"], description="train_rec")


## 6. Sous-tache Emotion-aligned Response Generation (Section 4.3)

Deux variantes sont fournies dans le depot ECR :

- **ECR[DialoGPT]** : `train_emp.py` puis `infer_emp.py`. On injecte le prompt emotion-aligned (Eq. 8) construit a partir des reviews IMDb filtrees (34,953 reviews / 4,092 films).
- **ECR[Llama 2-Chat]** : fine-tuning LoRA via LLaMA Board sur `src_emo/data/emo_data/llama_train.json` + `llama_test.json` (2,459 reviews / 1,553 films).

La fonction ci-dessous pilote la variante DialoGPT (la variante Llama est fine-tunee via un outil externe).

In [11]:
# Cellule 9 - Emotion-aligned Response Generation (DialoGPT)
def run_response_generation_training(cfg):
    if cfg["dry_run"]:
        print("DRY_RUN=True -> pipeline generation non lance.")
        return None
    src_emo = cfg["src_emo_dir"]
    redial = src_emo / "data" / "redial"
    redial_gen = src_emo / "data" / "redial_gen"
    ok = True

    # merge_rec.py lit ../../save/redial_rec/rec.json -> cwd doit etre redial_gen.
    rec_json = src_emo / "save" / "redial_rec" / "rec.json"
    if (not rec_json.exists()) or rec_json.stat().st_size == 0:
        print(f"[skip] merge_rec: fichier absent/vide ({rec_json}). train_rec n'a pas produit de recommandations.")
        return False
    ok &= _run(["python", "merge_rec.py"], cwd=redial_gen, description="merge_rec")
    # imdb_review_entity_filter.py importe dataset_dbpedia -> cwd = src_emo.
    ok &= _run(["python", "imdb_review_entity_filter.py"], cwd=src_emo, description="imdb_review_entity_filter")
    # process_empthetic.py lit movie_reviews_entities_filtered.json en local -> cwd = redial.
    ok &= _run(["python", "process_empthetic.py"], cwd=redial, description="process_empthetic")

    if cfg.get("use_pretrained_ckpt"):
        print("use_pretrained_ckpt=True -> train_emp.py ignore, on passe direct a l'inference.")
    else:
        # Article: per_device_batch=20, grad_accum=1 -> effective batch 20.
        batch, grad_accum = scaled_batch(cfg, per_device_batch=20, grad_accum=1)
        train_cmd = accelerate_launch_cmd(cfg, "train_emp.py") + [
            "--dataset", "redial",
            "--num_train_epochs", "15",
            "--gradient_accumulation_steps", str(grad_accum),
            "--ignore_pad_token_for_loss",
            "--per_device_train_batch_size", str(batch),
            "--per_device_eval_batch_size", "64",
            "--num_warmup_steps", "9965",
            "--context_max_length", "150",
            "--resp_max_length", "150",
            "--learning_rate", str(cfg["lr_dialogpt"]),
            "--num_workers", str(cfg.get("num_workers", 16)),
        ]
        if cfg.get("enable_torch_compile", False):
            train_cmd.append("--enable_torch_compile")
        ok &= _run(train_cmd, cwd=src_emo, description="train_emp")

    infer_cmd = accelerate_launch_cmd(cfg, "infer_emp.py") + [
        "--dataset", "redial_gen",
        "--split", "test",
        "--per_device_eval_batch_size", "256",
        "--context_max_length", "150",
        "--resp_max_length", "150",
        "--num_workers", str(cfg.get("num_workers", 16)),
    ]
    ok &= _run(infer_cmd, cwd=src_emo, description="infer_emp")
    return ok


## 7. Chargement des resultats (Tables 1-3)

`load_results_data` lit des CSV facultatifs dans `results/`. En l'absence de fichiers (cas le plus courant : `DRY_RUN=True`), on retombe sur les valeurs publiees par l'article afin que le notebook reste reproductible :

- Table 1 — recommandation objective (AUC, RT@n, R@n) ;
- Table 2 — generation subjective (Emo Int, Emo Pers, Log Pers, Info, Life) ;
- Table 3 — etude d'ablation (UniCRS, ECR[L], ECR[LS], ECR[LG], ECR).

In [12]:
# Cellule 10 - Chargement des metriques (repli sur les tables de l'article)
def _fallback_objective():
    return pd.DataFrame([
        {"Model": "KBRD",    "AUC": 0.503, "RT@1": 0.040, "RT@10": 0.182, "RT@50": 0.381, "R@1": 0.037, "R@10": 0.175, "R@50": 0.360},
        {"Model": "KGSF",    "AUC": 0.513, "RT@1": 0.043, "RT@10": 0.195, "RT@50": 0.383, "R@1": 0.040, "R@10": 0.182, "R@50": 0.361},
        {"Model": "RevCore", "AUC": 0.514, "RT@1": 0.054, "RT@10": 0.230, "RT@50": 0.410, "R@1": 0.046, "R@10": 0.209, "R@50": 0.390},
        {"Model": "UCCR",    "AUC": 0.499, "RT@1": 0.038, "RT@10": 0.208, "RT@50": 0.423, "R@1": 0.039, "R@10": 0.198, "R@50": 0.407},
        {"Model": "UniCRS",  "AUC": 0.506, "RT@1": 0.052, "RT@10": 0.229, "RT@50": 0.439, "R@1": 0.047, "R@10": 0.212, "R@50": 0.414},
        {"Model": "ECR",     "AUC": 0.541, "RT@1": 0.055, "RT@10": 0.238, "RT@50": 0.452, "R@1": 0.049, "R@10": 0.220, "R@50": 0.428},
    ])


def _fallback_subjective_llm():
    return pd.DataFrame([
        {"Model": "UniCRS",                 "Emo Int": 0.400, "Emo Pers": 0.942, "Log Pers": 0.793, "Info": 0.673, "Life": 2.241},
        {"Model": "GPT-3.5-turbo-instruct", "Emo Int": 1.706, "Emo Pers": 3.043, "Log Pers": 3.474, "Info": 2.975, "Life": 4.182},
        {"Model": "GPT-3.5-turbo",          "Emo Int": 2.215, "Emo Pers": 3.754, "Log Pers": 4.782, "Info": 4.147, "Life": 5.338},
        {"Model": "Llama 2-7B-Chat",        "Emo Int": 3.934, "Emo Pers": 6.030, "Log Pers": 5.886, "Info": 5.904, "Life": 7.129},
        {"Model": "ECR[DialoGPT]",          "Emo Int": 4.011, "Emo Pers": 4.878, "Log Pers": 4.736, "Info": 5.094, "Life": 5.906},
        {"Model": "ECR[Llama 2-Chat]",      "Emo Int": 6.826, "Emo Pers": 7.724, "Log Pers": 6.702, "Info": 7.653, "Life": 8.063},
    ])


def _fallback_subjective_human():
    return pd.DataFrame([
        {"Model": "UniCRS",                 "Emo Int": 0.947, "Emo Pers": 0.775, "Log Pers": 1.158, "Info": 0.380, "Life": 1.805},
        {"Model": "GPT-3.5-turbo-instruct", "Emo Int": 2.048, "Emo Pers": 2.555, "Log Pers": 3.265, "Info": 1.822, "Life": 3.648},
        {"Model": "GPT-3.5-turbo",          "Emo Int": 2.890, "Emo Pers": 3.678, "Log Pers": 5.323, "Info": 3.233, "Life": 5.125},
        {"Model": "Llama 2-7B-Chat",        "Emo Int": 4.432, "Emo Pers": 6.152, "Log Pers": 6.393, "Info": 5.713, "Life": 7.463},
        {"Model": "ECR[DialoGPT]",          "Emo Int": 5.097, "Emo Pers": 4.817, "Log Pers": 5.398, "Info": 4.628, "Life": 6.385},
        {"Model": "ECR[Llama 2-Chat]",      "Emo Int": 7.130, "Emo Pers": 7.575, "Log Pers": 7.403, "Info": 7.172, "Life": 8.468},
    ])


def _fallback_ablation():
    return pd.DataFrame([
        {"Model": "UniCRS",  "AUC": 0.506, "RT@10": 0.229, "RT@50": 0.439, "R@10": 0.212, "R@50": 0.414},
        {"Model": "ECR[L]",  "AUC": 0.535, "RT@10": 0.229, "RT@50": 0.444, "R@10": 0.213, "R@50": 0.420},
        {"Model": "ECR[LS]", "AUC": 0.541, "RT@10": 0.232, "RT@50": 0.442, "R@10": 0.216, "R@50": 0.420},
        {"Model": "ECR[LG]", "AUC": 0.535, "RT@10": 0.232, "RT@50": 0.453, "R@10": 0.216, "R@50": 0.428},
        {"Model": "ECR",     "AUC": 0.541, "RT@10": 0.238, "RT@50": 0.452, "R@10": 0.220, "R@50": 0.428},
    ])


def load_results_data(cfg):
    """Charge les metriques depuis `results/` ou retombe sur les tables de l'article."""
    r = cfg["results_dir"]
    r.mkdir(parents=True, exist_ok=True)
    pairs = [
        (r / cfg["objective_file"], _fallback_objective),
        (r / cfg["subjective_llm_file"], _fallback_subjective_llm),
        (r / cfg["subjective_human_file"], _fallback_subjective_human),
        (r / cfg["ablation_file"], _fallback_ablation),
    ]
    frames = []
    for path, fallback in pairs:
        if path.exists():
            print(f"[load] {path.name} depuis results/")
            frames.append(pd.read_csv(path))
        else:
            print(f"[load] {path.name} absent -> fallback article")
            frames.append(fallback())
    return tuple(frames)

In [13]:
# Cellule 11 - EDA : distribution de feedback, labels d'emotion, couverture de reviews
def build_dataset_eda(cfg):
    """Produit les dataframes d'EDA alignes avec la Section 5.1 de l'article."""
    df_feedback = pd.DataFrame({
        "feedback": ["like", "dislike", "not say"],
        "count":    [8110,   490,      1400],  # proportions 81.1 / 4.9 / 14.0 %
    })
    # Estimation des parts des 9 labels d'emotion (Section 4.1.1 - negative = 8%)
    df_emotions = pd.DataFrame({
        "emotion": ["like", "curious", "happy", "grateful", "negative",
                      "neutral", "nostalgia", "agreement", "surprise"],
        "share":   [28.0,   14.0,      12.0,    10.0,       8.0,
                     11.0,    5.0,       7.0,        5.0],  # approximation pedagogique
    })
    df_reviews = pd.DataFrame([
        {"backbone": "DialoGPT",         "reviews": 34953, "movies": 4092},
        {"backbone": "Llama 2-7B-Chat",  "reviews": 2459,  "movies": 1553},
    ])
    df_weights = pd.DataFrame([
        {"feedback": k, "weight": v} for k, v in cfg["feedback_weights"].items()
    ])
    return df_feedback, df_emotions, df_reviews, df_weights


def describe_eda(df_feedback, df_emotions, df_reviews):
    print("=== Feedback distribution (Table Section 5.1) ===")
    print(df_feedback)
    print("\n=== Emotion labels (Section 4.1.1) ===")
    print(df_emotions)
    print("\n=== IMDb review coverage (Section 5.1) ===")
    print(df_reviews)

In [14]:
# Cellule 12 - Analyse et table comparative
def evaluate_results(df_obj, df_subj_llm, df_subj_human):
    best_auc = df_obj.loc[df_obj["AUC"].idxmax()]
    best_rt10 = df_obj.loc[df_obj["RT@10"].idxmax()]
    best_life_llm = df_subj_llm.loc[df_subj_llm["Life"].idxmax()]
    best_life_human = df_subj_human.loc[df_subj_human["Life"].idxmax()]
    return pd.DataFrame([
        {"metric": "best_auc",        "model": best_auc["Model"],        "value": best_auc["AUC"]},
        {"metric": "best_rt10",       "model": best_rt10["Model"],       "value": best_rt10["RT@10"]},
        {"metric": "best_life_llm",   "model": best_life_llm["Model"],   "value": best_life_llm["Life"]},
        {"metric": "best_life_human", "model": best_life_human["Model"], "value": best_life_human["Life"]},
    ])


def build_comparison_table(df_obj, df_subj_llm, df_subj_human):
    llm = df_subj_llm.add_suffix("_llm").rename(columns={"Model_llm": "Model"})
    human = df_subj_human.add_suffix("_human").rename(columns={"Model_human": "Model"})
    return df_obj.merge(llm, on="Model", how="left").merge(human, on="Model", how="left")


def simulate_hyperparam_sweep(df_obj):
    """Petit balayage pedagogique sur le poids `like` (Appendix B)."""
    baseline = df_obj.set_index("Model").loc["ECR", "AUC"]
    sweep = pd.DataFrame({
        "like_weight": [1.0, 1.5, 2.0, 2.5, 3.0],
        "AUC":         [baseline - 0.015, baseline - 0.006, baseline, baseline - 0.004, baseline - 0.011],
    })
    return sweep

## 8. Pipeline principal (**une seule cellule d'import de donnees**)

Cette cellule enchaine la preparation legere (clone + archives + donnees ReDial) et le chargement des metriques. Avec `DRY_RUN=True`, aucun script d'entrainement n'est lance mais on obtient des DataFrames prets pour la visualisation.

In [ ]:
# Cellule 13 - IMPORT DES DONNEES (pipeline principal, cellule unique)
cfg = get_config()

# 1. Environnement ECR
clone_ecr_repo(cfg)
patch_ecr_compat(cfg)       # compat transformers >= 4.40 / pyg >= 2.5
download_external_assets(cfg)

# 2. Modeles de base HF (DialoGPT-small + RoBERTa) dans src_emo/save/
#    + installation optionnelle des poids pre-entraines (flag use_pretrained_ckpt)
if not cfg["dry_run"]:
    install_base_models(cfg)
    install_pretrained_ckpts(cfg)

# 3. Preparation des donnees ReDial (Section 4.1)
if prepare_redial_data(cfg) is False:
    raise RuntimeError("prepare_redial_data a echoue -> stop fail-fast")

# 4. Les entrainements lourds ne sont lances que si DRY_RUN=False.
#    Si use_pretrained_ckpt=True, chaque run_* saute son training au profit
#    des poids installes ci-dessus (mais prepare quand meme les donnees).
if run_emotional_semantic_fusion(cfg) is False:
    raise RuntimeError("run_emotional_semantic_fusion a echoue -> stop fail-fast")
if run_recommendation_training(cfg) is False:
    raise RuntimeError("run_recommendation_training a echoue -> stop fail-fast")
if run_response_generation_training(cfg) is False:
    raise RuntimeError("run_response_generation_training a echoue -> stop fail-fast")

# 5. Chargement des metriques (CSV reels ou repli article)
df_obj, df_subj_llm, df_subj_human, df_ablation = load_results_data(cfg)
df_feedback, df_emotions, df_reviews, df_weights = build_dataset_eda(cfg)

describe_eda(df_feedback, df_emotions, df_reviews)
df_obj.head()


ECR repo deja present: /workspace/ECR
[patch] AdamW -> torch.optim (train_pre.py): deja applique (idempotent)
[patch] AdamW -> torch.optim (train_rec.py): deja applique (idempotent)
[patch] AdamW -> torch.optim (train_emp.py): deja applique (idempotent)
[patch] model_gpt2.py imports: deja applique (idempotent)
[patch] dataset_dbpedia.py SparseTensor: deja applique (idempotent)
[patch] model_prompt.py RGCNConv call: deja applique (idempotent)
[patch] Accelerator fp16 kwarg (train_pre.py): deja applique (idempotent)
[patch] Accelerator fp16 kwarg (train_rec.py): deja applique (idempotent)
[patch] Accelerator fp16 kwarg (train_emp.py): deja applique (idempotent)
[patch] Accelerator fp16 kwarg (infer_emp.py): deja applique (idempotent)
[patch] config.py seed_torch: deja applique (idempotent)
[patch] accelerator.use_fp16 (train_pre.py): deja applique (idempotent)
[patch] accelerator.use_fp16 (train_emp.py): deja applique (idempotent)
[patch] accelerator.use_fp16 (infer_emp.py): deja appliqu

## 9. Compilation finale et visualisations comparatives

On recapitule les resultats (meilleurs scores, table comparative) puis on affiche tous les graphiques definis dans `src/viz/plots.py`. Les visualisations suivent les figures et tableaux de l'article :

- `plot_feedback_distribution`, `plot_feedback_weights`, `plot_emotion_label_distribution`, `plot_review_coverage` → Section 4.1 & 5.1 ;
- `plot_objective_metrics`, `plot_model_rankings`, `plot_ablation_study` → Tables 1 et 3 ;
- `plot_subjective_metrics`, `plot_subjective_radar`, `plot_llm_vs_human_correlation` → Table 2 & Section 5.6 ;
- `plot_training_loss`, `plot_hyperparam_sweep` → diagnostics d'entrainement et Appendix B.

In [ ]:
# Cellule 14 - Compilation finale : resume + visualisations
summary_df = evaluate_results(df_obj, df_subj_llm, df_subj_human)
comparison_df = build_comparison_table(df_obj, df_subj_llm, df_subj_human)

print("=== Summary ===")
print(summary_df)
print("\n=== Comparison table ===")
print(comparison_df.head())

# Dataset / EDA (Sections 4.1 et 5.1)
plot_feedback_distribution(df_feedback)
plot_feedback_weights(cfg["feedback_weights"])
plot_emotion_label_distribution(df_emotions)
plot_review_coverage(df_reviews)

# Recommandation (Table 1 + ablation Table 3)
plot_model_rankings(df_obj, metric="AUC")
plot_objective_metrics(df_obj)
plot_ablation_study(df_ablation)

# Generation de reponses (Table 2 + LLM vs Human)
plot_subjective_metrics(df_subj_llm, "Subjective metrics - LLM scorer (GPT-4-turbo)")
plot_subjective_metrics(df_subj_human, "Subjective metrics - Human annotators")
plot_subjective_radar(df_subj_llm, "ECR[Llama 2-Chat]", "Radar - ECR[Llama 2-Chat] (LLM scorer)")
plot_subjective_radar(df_subj_human, "ECR[Llama 2-Chat]", "Radar - ECR[Llama 2-Chat] (human)")
plot_llm_vs_human_correlation(df_subj_llm, df_subj_human)

# Diagnostics d'entrainement (si l'utilisateur a entraine les modeles)
history_path = cfg["results_dir"] / "training_history.csv"
if history_path.exists():
    df_history = pd.read_csv(history_path)
    plot_training_loss(df_history)
else:
    print(f"[info] {history_path.name} absent -> `plot_training_loss` ignore (disponible apres un vrai entrainement).")

# Sensibilite au poids `like` (Appendix B)
df_sweep = simulate_hyperparam_sweep(df_obj)
plot_hyperparam_sweep(df_sweep, param="like_weight", metric="AUC")